In [1]:
import sys
sys.path.insert(0, '..')
from src.search.engine import LegalSemanticSearchEngine
from src.loader import load_json

/home/gshjis/Python_projects/IA_NPA_auditor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Загружаем статьи
raw_D = load_json("/home/gshjis/Python_projects/IA_NPA_auditor/data/processed/const.json")
articles = [r["content"] for r in raw_D]

print(f"Загружено статей: {len(articles)}")


Загружено статей: 158


In [3]:
tagger = LegalSemanticSearchEngine(
    tags_filepath="/home/gshjis/Python_projects/IA_NPA_auditor/data/raw/base.json", 
    training_corpus=articles,
    tags_per_article=20
)

2026-03-16 11:04:46,816 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Загружено 77 тегов из /home/gshjis/Python_projects/IA_NPA_auditor/data/raw/base.json
2026-03-16 11:04:46,817 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Загрузка модели deepvk/USER2-base...
2026-03-16 11:04:46,817 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Попытка загрузки модели из локального кэша: deepvk/USER2-base...


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 5542.50it/s]


2026-03-16 11:04:47,265 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Модель загружена
2026-03-16 11:04:47,266 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Создано 77 описаний тегов
2026-03-16 11:04:47,267 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Загрузка эмбеддингов из data/cache/tag_embeddings.npy...
2026-03-16 11:04:47,268 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Тегирование обучающего корпуса...
2026-03-16 11:04:47,269 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Вычисление эмбеддингов для 158 статей...
2026-03-16 11:05:52,534 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Вычисление схожести с тегами...
2026-03-16 11:05:52,549 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Загрузка эмбеддингов из data/cache/article_embeddings.npy...
2026-03-16 11:05:52,560 - IA_NPA_auditor - INFO - ℹ️ ℹ️ IDF веса вычислены
2026-03-16 11:05:52,564 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Инициализация завершена


In [4]:
# ============================================================================
# ТЕСТ 1: Проверка тегов для проблемного запроса
# ============================================================================
art_44 = ""
for i in raw_D:
    if i["number"] == 44:
        art_44 = i["content"]
        break

tests = [t["content"] for t in raw_D[:5]]
tests.append("'Как наследуется квартира?'")
tests.append(art_44)
for query in tests:
    tags = tagger.get_tag_recommendations_formatted(query)
    print(tags)



2026-03-16 11:05:52,576 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Получение рекомендаций тегов для текста длиной 363...
2026-03-16 11:05:52,884 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Найдено 20 тегов

🔍 ТЕГИ ДЛЯ ЗАПРОСА: 'Республика Беларусь – унитарное демократическое социальное правовое государство. Республика Беларусь обладает верховенством и полнотой власти на своей территории, самостоятельно осуществляет внутреннюю и внешнюю политику. Республика Беларусь защищает свою независимость и территориальную целостность, конституционный строй, обеспечивает законность и правопорядок.'
----------------------------------------
   форма_правления                [████████████░░░░░░░░] 0.631
   язык                           [████████████░░░░░░░░] 0.603
   форма_устройства               [████████████░░░░░░░░] 0.600
   разделение_властей             [███████████░░░░░░░░░] 0.595
   верховенство_права             [███████████░░░░░░░░░] 0.593
   оборона_и_безопасность         [███████████░░░░░░░░░] 0.578
   правоохр

In [ ]:
# ============================================================================
# ТЕСТ 2: Поиск статей по запросу
# ============================================================================

query = "Каков правовой режим имущества, нажитого супругами во время брака, и как осуществляется раздел совместно нажитого имущества при разводе с учетом прав наследования?"
print("\n" + "=" * 80)
print(f"🔍 ПОИСК СТАТЕЙ ПО ЗАПРОСУ: {query}".center(80))
print("=" * 80)

results = tagger.find_articles_by_new_sentence(query, k=20)

for rank, res in enumerate(results, 1):
    # Иконка в зависимости от score
    if res['score'] >= 0.6:
        icon = "🟢🔝"
    elif res['score'] >= 0.5:
        icon = "🟡"
    elif res['score'] >= 0.4:
        icon = "🔸"
    else:
        icon = "⚪"
    
    print(f"\n{icon} [{rank}] РЕЛЕВАНТНОСТЬ: {res['score']:.3f}")
    
    # Показываем начало статьи
    text_preview = res['text'][:200] + "..." if len(res['text']) > 200 else res['text']
    print(f"   {text_preview}")
    
    # Отмечаем статью 44, если она найдена
    if "наследование" in res['text'].lower():
        print("   ✅ ЭТО СТАТЬЯ 44!")



🔍 ПОИСК СТАТЕЙ ПО ЗАПРОСУ: Каков правовой режим имущества, нажитого супругами во время брака, и как осуществляется раздел совместно нажитого имущества при разводе с учетом прав наследования?
2026-03-16 11:06:52,199 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Поиск статей для запроса: 'Каков правовой режим имущества, нажитого супругами...'
2026-03-16 11:06:52,201 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Получение рекомендаций тегов для текста длиной 163...
2026-03-16 11:06:52,402 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Найдено 20 тегов
2026-03-16 11:06:52,410 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Найдено 50 статей

🟢🔝 [1] РЕЛЕВАНТНОСТЬ: 0.623
   Дела в судах рассматриваются судьями единолично, а в предусмотренных законом случаях – коллегиально.

🟡 [2] РЕЛЕВАНТНОСТЬ: 0.559
   Разбирательство дел во всех судах открытое. Слушание дел в закрытом судебном заседании допускается лишь в случаях, определенных законом, с соблюдением всех правил судопроизводства.

🟡 [3] РЕЛЕВАНТНОСТЬ: 0.555
   Каждый обязан уважать достоинство, пр